In [1]:
from scipy.spatial import KDTree
from scipy.spatial.transform import Rotation as R
import numpy as np
import plotly.graph_objects as go
from IPython.display import clear_output
import time

rng = np.random.default_rng(0)

## Now we create a test set 

In [18]:
N = 200
points = rng.uniform(-10, 10, size=(N, 3))
rotation = R.from_euler('xyz', [0.1, 0.2, 0.3]).as_matrix()
translation = np.array([50.0, 2.5, -3.0])

source = points
target = points @ rotation.T + translation

In [3]:
def plot_graph(target, pc, title=''):
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=target[:, 0], y=target[:, 1], z=target[:, 2],
                               mode='markers', marker=dict(size=4), name='target'))
    fig.add_trace(go.Scatter3d(x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
                               mode='markers', marker=dict(size=4), name='pc'))
    fig.update_layout(title=title, scene=dict(aspectmode='data'))
    fig.show()

In [19]:
plot_graph(target, source, title='Initial Point Clouds')

In [5]:
def best_fit_transform(A, B):
    mu_a = np.mean(A, axis=0)
    mu_b = np.mean(B, axis=0)

    A_c = A - mu_a
    B_c = B - mu_b

    H = A_c.T @ B_c
    U, S, Vt = np.linalg.svd(H)

    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[2, :] *= -1
        R = Vt.T @ U.T

    t = mu_b - R @ mu_a
    return R, t


### here we observe that svd works for our case, but if there poitns were simply random like in lidar, we will notfare well.

In [20]:
R_est, t_est = best_fit_transform(source, target)
print(np.allclose(R_est, rotation), np.allclose(t_est, translation))

True True


In [21]:
shuffled_target = target[rng.permutation(N)]
R_bad, t_bad = best_fit_transform(source, shuffled_target)
print(np.allclose(R_bad, rotation), np.allclose(t_bad, translation))

False False


In [33]:
def icp(source, target, max_iterations=50, tolerance=1e-5):
    tree = KDTree(target)
    pc = source.copy()
    R_total = np.eye(3)
    t_total = np.zeros(3)
    prev_error = np.inf

    for i in range(max_iterations):
        distance, idx = tree.query(pc)
        matched = target[idx]

        R_step, t_step = best_fit_transform(pc, matched)

        pc = pc@R_step.T + t_step

        t_total = R_step @ t_total + t_step
        R_total = R_step @ R_total

        clear_output(wait=True)
        plot_graph(target, pc, f'iteration {i}, error {distance.mean():.4f}')

        error = distance.mean()
        if abs(prev_error - error) < tolerance:
            break
        prev_error = error 

        time.sleep(0.1)

    return R_total, t_total, pc
    

In [34]:
R_est, t_est, aligned = icp(source, shuffled_target)

## Noisy lidar points 


In [36]:
sigma = 0.05
noise = rng.normal(0, sigma, size=target.shape)

target_noisy = target + noise
shuffled_target = target_noisy[rng.permutation(N)]

In [41]:
R_est, t_est, aligned = icp(source, shuffled_target)